# M9.16 · Viola–Jones — Haar öznitelikleri, AdaBoost ve kaskad

İlk gerçek zamanlı yüz dedektörü: integral görüntüyle sabit sürede Haar benzeri öznitelikler, AdaBoost ile binlerce öznitelikten birkaç onunu seçmek ve çoğu pencereyi ilk aşamalarda eleyen kaskad. Saf Python'da integral görüntü ve 5 tür Haar özniteliği (`skimage` ile aynı işaretler); NumPy'da binlerce özniteliği bütün veri kümesi üzerinde tek seferde hesaplayıp karar kütüklerini (decision stump) sıralama + kümülatif toplamla buluyorsun; PyTorch'ta tüm öznitelikler için aynı anda en iyi kütüğü arayan AdaBoost; en sonda kaskadı kurup bir sahnedeki tüm pencereleri vektörel olarak tarıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-16-viola-jones.html](https://m-gonul.github.io/pytorch-lab/m9-16-viola-jones.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_16_viola_jones.py
CHECKS = {
 "b": "import numpy as np\nfrom skimage import data\nfrom skimage.feature import haar_like_feature, haar_like_feature_coord\nfrom skimage.transform import integral_image\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/skimage kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"skimage\", \"cv2\", \"scipy\"}))\n    img = data.lfw_subset()[3]\n\n    def ii():\n        got = np.array(need(ns, \"integral\")(img.tolist()))\n        want = np.pad(img, ((1, 0), (1, 0))).cumsum(0).cumsum(1)\n        close(got, want, atol=1e-9, what=\"integral\")\n        close(need(ns, \"rect_sum\")(got.tolist(), [(3, 4), (10, 12)]), img[3:11, 4:13].sum(), atol=1e-9, what=\"rect_sum\")\n    R.test(\"integral ve rect_sum\", ii)\n\n    for kind in (\"type-2-x\", \"type-2-y\", \"type-3-x\", \"type-3-y\", \"type-4\"):\n        def t(kind=kind):\n            coords, types = haar_like_feature_coord(25, 25, kind)\n            idx = np.random.default_rng(len(kind)).choice(len(coords), 300, replace=False)\n            c = coords[idx]; ty = types[idx]\n            want = haar_like_feature(integral_image(img), 0, 0, 25, 25, feature_type=ty, feature_coord=c)\n            got = need(ns, \"features_at\")(img.tolist(), [list(map(tuple, map(lambda r: list(map(tuple, r)), x))) for x in c], list(ty))\n            close(np.array(got), want, atol=1e-9, what=kind)\n        R.test(f\"haar = skimage ({kind})\", t)\n",
 "c": "import numpy as np\nfrom skimage import data\nfrom skimage.feature import haar_like_feature, haar_like_feature_coord\nfrom skimage.transform import integral_image\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yalnız rect_table'da\", lambda: (max_loops(src, 0, func=\"haar_matrix\"), max_loops(src, 0, func=\"best_stump\")))\n    X = data.lfw_subset()\n    rng = np.random.default_rng(0)\n    coords, kinds = [], []\n    for k in (\"type-2-x\", \"type-2-y\", \"type-3-x\", \"type-3-y\", \"type-4\"):\n        c, t = haar_like_feature_coord(25, 25, k)\n        idx = rng.choice(len(c), 60, replace=False)\n        coords += list(c[idx]); kinds += list(t[idx])\n    coords_a = np.empty(len(coords), object); coords_a[:] = coords\n    kinds_a = np.array(kinds, object)\n\n    def hm():\n        Rt, St = need(ns, \"rect_table\")(coords, kinds)\n        F = need(ns, \"haar_matrix\")(X[:20], Rt, St)\n        ensure(F.shape == (20, 300), f\"şekil {F.shape}\")\n        for i in (0, 7, 19):\n            want = haar_like_feature(integral_image(X[i]), 0, 0, 25, 25, feature_type=kinds_a, feature_coord=coords_a)\n            close(F[i], want, atol=1e-9, what=f\"görüntü {i}\")\n    R.test(\"haar_matrix = skimage (300 öznitelik, 20 görüntü)\", hm)\n\n    def bs():\n        for trial in range(20):\n            f = np.round(rng.normal(size=40), 1); y = rng.integers(0, 2, 40).astype(float); w = rng.random(40); w /= w.sum()\n            err, thr, pol = need(ns, \"best_stump\")(f, y, w)\n            best = min(((w * (((p * f) <= (p * t)).astype(float) != y)).sum(), t, p) for t in np.unique(f) for p in (1, -1))\n            close(err, best[0], atol=1e-12, what=\"en küçük ağırlıklı hata\")\n            pred = ((pol * f) <= (pol * thr)).astype(float)\n            close((w * (pred != y)).sum(), err, atol=1e-12, what=\"döndürülen eşik bu hatayı vermeli\")\n    R.test(\"best_stump = kaba kuvvet (20 rastgele durum)\", bs)\n",
 "d": "import numpy as np\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    X = data.lfw_subset()\n    y = np.r_[np.ones(100), np.zeros(100)]\n    Rt, St, At = ns[\"feature_table\"](3000, 0)\n    F = torch.tensor(ns[\"normalized_features\"](X, Rt, St, At))\n    rng = np.random.default_rng(0)\n    perm = rng.permutation(200); tr, te = perm[:150], perm[150:]\n\n    def bs():\n        w = torch.tensor(rng.random(150)); w /= w.sum()\n        yt = torch.tensor(y[tr])\n        err, thr, pol = need(ns, \"best_stumps\")(F[tr][:, :50], yt, w)\n        for j in range(0, 50, 7):\n            f = F[tr][:, j].numpy()\n            best = min((w.numpy() * (((p * f) <= (p * t)).astype(float) != y[tr])).sum() for t in np.unique(f) for p in (1, -1))\n            close(float(err[j]), best, atol=1e-12, what=f\"öznitelik {j}\")\n    R.test(\"best_stumps = kaba kuvvet\", bs)\n\n    def ab():\n        H = need(ns, \"adaboost\")(F[tr], torch.tensor(y[tr]), 30)\n        ensure(len(H) == 30, \"30 zayıf sınıflandırıcı\")\n        rows = []\n        for k in (1, 3, 10, 30):\n            s, half = need(ns, \"strong_score\")(F[te], H, k)\n            acc = float(((s >= half).double().numpy() == y[te]).mean())\n            rows.append([k, f\"{acc:.2f}\"])\n        ensure(float(rows[-1][1]) >= 0.9, f\"30 öznitelikle test doğruluğu {rows[-1][1]}\")\n        j = H[0][0]\n        r = Rt[j][:, :4]\n        R.table(\"lfw_subset (150 eğitim / 50 test), 3000 rastgele öznitelik\", [\"öznitelik sayısı\", \"test doğruluğu\"], rows,\n                note=f\"İlk seçilen öznitelik dikdörtgenleri (satır, sütun): {[tuple(x) for x in r if x[2] > 0]} — genelde göz bandı / yanak karşıtlığı.\")\n    R.test(\"AdaBoost\", ab)\n",
 "e": "import time\nimport numpy as np\nimport torch\nfrom skimage import data\nfrom labcheck import need, ensure, fmt_time\n\n\ndef scene(X, faces_idx, bg_idx, seed=0):\n    rng = np.random.default_rng(seed)\n    img = np.zeros((200, 200))\n    cells = [(r, c) for r in range(8) for c in range(8)]\n    for (r, c) in cells:\n        p = X[rng.choice(bg_idx)]\n        img[r * 25:(r + 1) * 25, c * 25:(c + 1) * 25] = p[:, ::-1] if rng.random() < 0.5 else p\n    truth = []\n    for i, (r, c) in zip(faces_idx, [(1, 1), (2, 5), (5, 2), (6, 6)]):\n        img[r * 25:(r + 1) * 25, c * 25:(c + 1) * 25] = X[i]\n        truth.append((c * 25, r * 25))\n    return img, truth\n\n\ndef check(ns, R):\n    X = data.lfw_subset()\n    y = np.r_[np.ones(100), np.zeros(100)]\n    perm = np.random.default_rng(0).permutation(200); tr, te = perm[:150], perm[150:]\n    Rt, St, At = ns[\"feature_table\"](3000, 0)\n    F = torch.tensor(ns[\"normalized_features\"](X, Rt, St, At))\n    H = ns[\"train\"](F[tr], torch.tensor(y[tr]), 30)\n\n    def mc():\n        cas = need(ns, \"make_cascade\")(F[tr][y[tr] == 1], H)\n        ensure([k for k, _ in cas] == [2, 10, 30], \"aşama boyutları\")\n        for k, thr in cas:\n            s = sum(a * ((p * F[tr][y[tr] == 1][:, j]) <= (p * t)).double() for j, t, p, a in H[:k])\n            ensure(bool((s >= thr).all()), f\"aşama {k}: tüm eğitim yüzleri geçmeli\")\n    R.test(\"make_cascade: eğitim yüzlerinin hepsi geçer\", mc)\n\n    def sc():\n        cas = ns[\"make_cascade\"](F[tr][y[tr] == 1], H)\n        img, truth = scene(X, [i for i in te if y[i] == 1][:4], [i for i in te if y[i] == 0])\n        t0 = time.perf_counter()\n        boxes, counts = need(ns, \"scan\")(img, Rt, St, At, H, cas)\n        t1 = time.perf_counter()\n        dets = need(ns, \"nms\")(boxes)\n        found = sum(any(abs(x - tx) <= 3 and abs(y_ - ty) <= 3 for x, y_, _ in dets) for tx, ty in truth)\n        ensure(counts[0] == 176 * 176, f\"ilk aşamaya {counts[0]} pencere girmeli (176²)\")\n        ensure(counts[1] < counts[0] * 0.6, \"ilk aşama pencerelerin çoğunu elemeli\")\n        ensure(found >= 3, f\"4 yüzden {found} tanesi bulundu\")\n        evals = sum(c * k for c, (k, _) in zip(counts, cas))\n        R.table(\"200 × 200 sahne (8 × 8 yama, 4'ü görülmemiş yüz)\", [\"aşama\", \"öznitelik\", \"giren pencere\"], [[i + 1, k, c] for i, ((k, _), c) in enumerate(zip(cas, counts))],\n                note=f\"Bulunan yüz {found}/4, NMS sonrası toplam tespit {len(dets)}. Öznitelik değerlendirmesi: kaskadla {evals:,}, tek aşamalı 30 öznitelikle {counts[0] * 30:,} ({counts[0] * 30 / evals:.1f}× tasarruf). Süre {fmt_time(t1 - t0)}.\")\n    R.test(\"tarama + kaskad + NMS\", sc)\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def ws():\n        img = data.camera()[:100, :120]\n        m, s = need(ns, \"window_stats\")(img, 24)\n        ensure(m.shape == (77, 97), f\"şekil {m.shape}\")\n        win = np.lib.stride_tricks.sliding_window_view(img.astype(float), (24, 24))\n        close(m, win.mean((-2, -1)), atol=1e-9, what=\"ortalama\")\n        close(s, win.std((-2, -1)), atol=1e-6, what=\"std\")\n    R.test(\"window_stats\", ws)\n\n    def dt():\n        rows = []\n        for H, W in [(16, 16), (640, 480), (1920, 1080), (8000, 8000)]:\n            a, b = need(ns, \"min_dtype\")(H, W)\n            rows.append([f\"{W} × {H}\", np.dtype(a).name, np.dtype(b).name])\n        ensure(rows[0][1] == \"uint16\" and rows[1][1] == \"uint32\" and rows[1][2] == \"uint64\" and rows[3][1] == \"uint64\", f\"{rows}\")\n        R.table(\"Integral görüntü için gereken tip (8 bit giriş)\", [\"boyut\", \"ii\", \"kare ii\"], rows,\n                note=\"uint8 görüntünün kümülatif toplamını uint8'de tutmak (alıştırma 5) neredeyse her pikselde taşar.\")\n    R.test(\"min_dtype\", dt)\n",
 "x2": "from skimage.feature import haar_like_feature_coord\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    def t():\n        rows = []\n        for kind, (nx, ny) in {\"type-2-x\": (2, 1), \"type-2-y\": (1, 2), \"type-3-x\": (3, 1), \"type-3-y\": (1, 3), \"type-4\": (2, 2)}.items():\n            for W, H in [(24, 24), (19, 19), (25, 20)]:\n                want = len(haar_like_feature_coord(W, H, kind)[0])\n                got = need(ns, \"count_type\")(W, H, nx, ny)\n                ensure(got == want, f\"{kind} {W}×{H}: {got}, skimage {want}\")\n            rows.append([kind, need(ns, \"count_type\")(24, 24, nx, ny)])\n        ensure(need(ns, \"total_features\")(24, 24) == 162336, \"24 × 24 toplam 162 336\")\n        R.table(\"24 × 24 pencere\", [\"tür\", \"öznitelik\"], rows + [[\"toplam\", 162336]])\n    R.test(\"öznitelik sayıları = skimage\", t)\n",
 "x3": "import numpy as np\nfrom sklearn.metrics import roc_auc_score, roc_curve\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def cr():\n        D, F = need(ns, \"cascade_rates\")([(0.99, 0.3)] * 10)\n        close(D, 0.99 ** 10, what=\"D\"); close(F, 0.3 ** 10, what=\"F\")\n        ensure(need(ns, \"stages_needed\")(0.3, 1e-6) == 12 and ns[\"stages_needed\"](0.5, 1e-3) == 10, \"aşama sayısı\")\n    R.test(\"kaskad aritmetiği\", cr)\n\n    def rc():\n        rng = np.random.default_rng(0)\n        for _ in range(5):\n            y = rng.integers(0, 2, 300); s = np.round(rng.normal(y * 1.2, 1.0), 1)\n            fpr, tpr, auc = need(ns, \"roc\")(s, y)\n            close(auc, roc_auc_score(y, s), atol=1e-12, what=\"AUC\")\n            f2, t2, _ = roc_curve(y, s, drop_intermediate=False)\n            close(np.array(fpr), f2, atol=1e-12, what=\"fpr\"); close(np.array(tpr), t2, atol=1e-12, what=\"tpr\")\n    R.test(\"roc = sklearn\", rc)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_16_viola_jones as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Haar benzeri öznitelikler:** bitişik dikdörtgenlerin toplamları arasındaki fark. Beş tür (scikit-image adlarıyla): `type-2-x`, `type-2-y` (kenar), `type-3-x`, `type-3-y` (çizgi), `type-4` (çapraz). İşaretler: iki dikdörtgende $r_2-r_1$, üçte $r_2-(r_1+r_3)$ (orta − kenarlar), dörtte $(r_2+r_4)-(r_1+r_3)$ (dikdörtgenler sol üst, sağ üst, sağ alt, sol alt sırasıyla). Yüzdeki göz bandı yanaklardan koyu, burun köprüsü gözlerden açık: bu karşıtlıkları yakalarlar. 24 × 24 pencerede tüm konum ve boyutlarda **162 336** öznitelik vardır.

**Integral görüntü:** $ii(y,x)=\sum_{y'<y,\,x'<x}f(y',x')$; dikdörtgen toplamı 4 erişim. Pencere başına normalizasyon için ayrıca $f^2$'nin integral görüntüsü: $\sigma^2=E[f^2]-\mu^2$.

**AdaBoost (Viola–Jones biçimi):**

1. Ağırlıklar: pozitiflere $1/2m$, negatiflere $1/2l$.
2. Her tur $t$: ağırlıkları normalize et; her öznitelik için en iyi **karar kütüğü** $h(x)=[p\,f(x)<p\,\theta]$ (eşik $\theta$, polarite $p$) ve ağırlıklı hatası $\epsilon$; en küçük $\epsilon$'lusunu seç.
3. $\beta_t=\epsilon_t/(1-\epsilon_t)$; doğru sınıflananların ağırlığını $\beta_t$ ile çarp.
4. Güçlü sınıflandırıcı: $\sum_t\alpha_th_t(x)\ge\frac12\sum_t\alpha_t$, $\alpha_t=\log1/\beta_t$.

**Kaskad:** aşamalar sırayla; bir aşama "değil" derse pencere hemen reddedilir. Aşama eşikleri tespit oranı ~%99'u koruyacak şekilde düşürülür. 10 aşama, $d_i=0.99$, $f_i=0.3$: $D=0.90$, $F=6\times10^{-6}$. İlk aşamalar 2–10 öznitelikle pencerelerin çoğunu eler.

**Tarama:** pencereyi tüm konum ve ölçeklerde kaydır (görüntüyü küçültmek yerine öznitelikleri büyüt), örtüşen tespitleri birleştir. Artıları: gerçek zamanlı, aydınlatmaya görece dayanıklı. Eksileri: yalnız dik ve önden yüzler, dönmeye değişmez değil; bugün CNN tabanlı dedektörler geçti.

## 1 · Elle

<!-- çözüm -->

Kutu toplamı: $500-120-150+40=270$.

Kaskadın gücü: her aşama yalnız %30 yanlış pozitife izin verse de 10 aşama sonunda milyonda 6.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Integral görüntü ve öznitelik**

- `box`: $ii(9,9)=500$, $ii(3,9)=120$, $ii(9,3)=150$, $ii(3,3)=40$: dikdörtgen toplamı
- `f2`: İki dikdörtgenli öznitelik: sol 270, sağ 330 ($r_2-r_1$)
- `cnt`: 24 × 24 pencerede 5 türün toplam öznitelik sayısı

**AdaBoost ve kaskad**

- `beta`: Ağırlıklı hata $\epsilon=0.2$: $\beta$
- `alpha`: $\alpha=\ln(1/\beta)$
- `wu`: Doğru sınıflanan örneğin ağırlığı 0.1 → (normalizasyondan önce)
- `D`: 10 aşama, $d_i=0.99$: toplam tespit oranı
- `F`: 10 aşama, $f_i=0.3$: toplam yanlış pozitif × 10⁶

In [ ]:
cevaplar = dict(
    box=None,
    f2=None,
    cnt=None,
    beta=None,
    alpha=None,
    wu=None,
    D=None,
    F=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"box":[270,0],"f2":[60,0],"cnt":[162336,0],"beta":[0.25,0],"alpha":[1.386294,0.0001],"wu":[0.025,0],"D":[0.904382,0.0001],"F":[5.9049,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. Dikdörtgen biçimi scikit-image'inki gibi: `[(r0, c0), (r1, c1)]`, **uçlar dahil**.

- `integral(img)` — `(H+1) × (W+1)` liste, ilk satır ve sütun 0.
- `rect_sum(ii, rect)` — 4 erişim.
- `haar(ii, rects, kind)` — `kind` ∈ `"type-2-x"`, `"type-2-y"`, `"type-3-x"`, `"type-3-y"`, `"type-4"`; giriş metnindeki işaretler.
- `features_at(img, coords, kinds)` — bir görüntü için öznitelik listesi.

Kontrol, `skimage.feature.haar_like_feature` ile aynı değerleri bekler.

In [ ]:
%%writefile stage_b.py
def integral(img):
    ...


def rect_sum(ii, rect):
    ...


def haar(ii, rects, kind):
    ...


def features_at(img, coords, kinds):
    ...


ii = integral([[1, 2], [3, 4]])
print(ii, rect_sum(ii, [(0, 0), (1, 1)]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok** (öznitelik tanımlarını diziye çevirme hariç).

- `rect_table(coords, kinds)` → `(R, S)`: `R (K, 4, 4)` her özniteliğin en fazla 4 dikdörtgeni `[r0, c0, r1+1, c1+1]` (eksikler 0), `S (K, 4)` işaretler (eksikler 0). Bu yardımcı için döngü serbest.
- `haar_matrix(X, R, S)` — `X (N, H, W)` → `(N, K)`: integral görüntüler `(N, H+1, W+1)`, dört köşe `I[:, r1, c1] − I[:, r0, c1] − I[:, r1, c0] + I[:, r0, c0]` gelişmiş indekslemeyle `(N, K, 4)`, işaretlerle topla.
- `best_stump(f, y, w)` — tek öznitelik için en iyi eşik: değerleri sırala, `cumsum` ile "eşiğin altı yüz" ve "eşiğin altı yüz değil" hatalarını **tüm eşikler için** hesapla, en küçüğü. `(err, thr, pol)`: tahmin `pol·f ≤ pol·thr` ise yüz (1). Bölünme yalnız farklı iki değer arasında olabilir: **eşit değerler** varsa her değerin yalnız son konumu geçerlidir (yoksa aynı değerli örneklerin bir kısmı yanlış tarafa sayılır). Eşik olarak ardışık iki farklı değerin **ortasını** döndür (en sondaki için son değer + 1): böylece iki polarite de aynı bölünmeyi verir.

In [ ]:
%%writefile stage_c.py
import numpy as np


def rect_table(coords, kinds):
    ...


def haar_matrix(X, R, S):
    ...


def best_stump(f, y, w):
    ...


from skimage import data
from skimage.feature import haar_like_feature_coord
c, k = haar_like_feature_coord(25, 25, "type-2-x")
R, S = rect_table(c[:10], k[:10])
print(haar_matrix(data.lfw_subset()[:3], R, S).shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Tüm öznitelikler için **aynı anda** en iyi kütük ve AdaBoost (PyTorch).

- `best_stumps(F, y, w)` — `F (N, K)` `float64`, `y (N,)` 0/1, `w (N,)`: her sütunu sırala (`argsort(0)`), sıralı ağırlıklarla kümülatif toplamlar (c'deki gibi eşit değerlerde yalnız son konum); `e1 = S_neg + (T_pos − S_pos)` (eşiğin altı yüz), `e2 = S_pos + (T_neg − S_neg)` (eşiğin altı yüz değil); her öznitelik için en küçük hata, eşik (c'deki gibi ardışık değerlerin ortası) ve polarite (`e1 ≤ e2` ise +1). `(err (K,), thr (K,), pol (K,))`.
- `adaboost(F, y, T)` — Viola–Jones AdaBoost; `[(j, thr, pol, alpha), …]`.
- `strong_score(F, H, k=None)` — ilk `k` zayıf sınıflandırıcının $\sum\alpha_th_t$'si ve eşiği $\frac12\sum\alpha_t$: `(score (N,), half)`.

Veri: `skimage.data.lfw_subset()` (100 yüz, 100 yüz olmayan, 25 × 25); öznitelikler her pencerenin ortalama ve std'siyle normalize edilmiş Haar değerleri (hazır yardımcı).

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch
from skimage import data
from skimage.feature import haar_like_feature_coord

SIGNS = {"type-2-x": (-1, 1), "type-2-y": (-1, 1), "type-3-x": (-1, 1, -1), "type-3-y": (-1, 1, -1), "type-4": (-1, 1, -1, 1)}


def feature_table(K=5000, seed=0, size=25):
    """Rastgele K öznitelik: R (K,4,4) [r0,c0,r1+1,c1+1], S (K,4) işaret, A (K,) Σ işaret·alan."""
    rng = np.random.default_rng(seed)
    coords, kinds = [], []
    for k in SIGNS:
        c, t = haar_like_feature_coord(size, size, k)
        coords += list(c); kinds += list(t)
    sel = rng.choice(len(coords), K, replace=False)
    R = np.zeros((K, 4, 4), np.int64); S = np.zeros((K, 4))
    for j, i in enumerate(sel):
        for q, ((r0, c0), (r1, c1)) in enumerate(coords[i]):
            R[j, q] = [r0, c0, r1 + 1, c1 + 1]; S[j, q] = SIGNS[kinds[i]][q]
    A = (S * (R[..., 2] - R[..., 0]) * (R[..., 3] - R[..., 1])).sum(1)
    return R, S, A


def normalized_features(X, R, S, A):
    """Pencere başına (f − μ·A)/σ."""
    I = np.pad(X.astype(np.float64), ((0, 0), (1, 0), (1, 0))).cumsum(1).cumsum(2)
    r0, c0, r1, c1 = R[..., 0], R[..., 1], R[..., 2], R[..., 3]
    F = ((I[:, r1, c1] - I[:, r0, c1] - I[:, r1, c0] + I[:, r0, c0]) * S).sum(-1)
    mu = X.reshape(len(X), -1).mean(1, keepdims=True); sd = X.reshape(len(X), -1).std(1, keepdims=True) + 1e-8
    return (F - mu * A) / sd


def best_stumps(F, y, w):
    ...


def adaboost(F, y, T):
    ...


def strong_score(F, H, k=None):
    ...


X = data.lfw_subset()
R, S, A = feature_table(500)
F = torch.tensor(normalized_features(X, R, S, A))
y = torch.tensor(np.r_[np.ones(100), np.zeros(100)])
print(adaboost(F, y, 3))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Kaskad ve vektörel tarama. Hazır verilenler: d'deki eğitim (`train`), öznitelik tablosu.

- `make_cascade(F_pos, H, sizes=(2, 10, 30))` — aşama $i$, ilk `sizes[i]` zayıf sınıflandırıcıdır; eşiği eğitim yüzlerinin **hepsini** geçirecek şekilde `min(score)` (ve $\frac12\sum\alpha$'yı aşmayacak şekilde) seçilir. `[(k, thr), …]`.
- `scan(img, R, S, A, H, cascade, win=25)` → `(boxes, counts)`: tüm `(y, x)` pencereleri (adım 1); her pencere için ortalama/std (integral ve kare integral), öznitelik değerleri `(Nw, K)` gelişmiş indekslemeyle; **aşama aşama** yalnız hayatta kalan pencerelerle devam; `counts` her aşamaya giren pencere sayısı; `boxes` son aşamayı geçen `(x, y, score)` (score: son aşamanın skoru). Aşama döngüsü serbest.
- `nms(boxes, r=12)` — skora göre azalan sırada, kabul edilmiş bir kutuya `r`'den yakın merkezleri at.

In [ ]:
%%writefile stage_e.py
import numpy as np
import torch
from skimage import data
from skimage.feature import haar_like_feature_coord

# --- d'den hazır: öznitelik tablosu, normalize öznitelikler, AdaBoost ---
SIGNS = {"type-2-x": (-1, 1), "type-2-y": (-1, 1), "type-3-x": (-1, 1, -1), "type-3-y": (-1, 1, -1), "type-4": (-1, 1, -1, 1)}


def feature_table(K=3000, seed=0, size=25):
    rng = np.random.default_rng(seed)
    coords, kinds = [], []
    for k in SIGNS:
        c, t = haar_like_feature_coord(size, size, k)
        coords += list(c); kinds += list(t)
    sel = rng.choice(len(coords), K, replace=False)
    R = np.zeros((K, 4, 4), np.int64); S = np.zeros((K, 4))
    for j, i in enumerate(sel):
        for q, ((r0, c0), (r1, c1)) in enumerate(coords[i]):
            R[j, q] = [r0, c0, r1 + 1, c1 + 1]; S[j, q] = SIGNS[kinds[i]][q]
    A = (S * (R[..., 2] - R[..., 0]) * (R[..., 3] - R[..., 1])).sum(1)
    return R, S, A


def normalized_features(X, R, S, A):
    I = np.pad(X.astype(np.float64), ((0, 0), (1, 0), (1, 0))).cumsum(1).cumsum(2)
    r0, c0, r1, c1 = R[..., 0], R[..., 1], R[..., 2], R[..., 3]
    F = ((I[:, r1, c1] - I[:, r0, c1] - I[:, r1, c0] + I[:, r0, c0]) * S).sum(-1)
    mu = X.reshape(len(X), -1).mean(1, keepdims=True); sd = X.reshape(len(X), -1).std(1, keepdims=True) + 1e-8
    return (F - mu * A) / sd


def train(F, y, T):
    y = y.double()
    w = torch.where(y == 1, 0.5 / y.sum(), 0.5 / (1 - y).sum())
    H = []
    for _ in range(T):
        w = w / w.sum()
        o = F.argsort(0); Fs = torch.gather(F, 0, o); ys, ws = y[o], w[o]
        Sp, Sn = torch.cumsum(ws * ys, 0), torch.cumsum(ws * (1 - ys), 0)
        e1, e2 = Sn + ((w * y).sum() - Sp), Sp + ((w * (1 - y)).sum() - Sn)
        last = torch.cat([Fs[1:] != Fs[:-1], torch.ones(1, F.shape[1], dtype=torch.bool)])
        e1, e2 = torch.where(last, e1, torch.inf), torch.where(last, e2, torch.inf)
        mid = torch.cat([(Fs[:-1] + Fs[1:]) / 2, Fs[-1:] + 1])
        m1, i1 = e1.min(0); m2, i2 = e2.min(0)
        err = torch.minimum(m1, m2); j = int(err.argmin())
        pol = 1 if m1[j] <= m2[j] else -1
        thr = float(mid[i1[j] if pol == 1 else i2[j], j])
        beta = max(float(err[j]), 1e-10) / (1 - float(err[j]))
        h = ((pol * F[:, j]) <= (pol * thr)).double()
        w = w * torch.where(h == y, beta, 1.0)
        H.append((j, thr, pol, float(np.log(1 / beta))))
    return H


# --- senin kodun ---
def make_cascade(F_pos, H, sizes=(2, 10, 30)):
    ...


def scan(img, R, S, A, H, cascade, win=25):
    ...


def nms(boxes, r=12):
    ...


print("kaskad")

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Pencere varyans normalizasyonu ve integral görüntünün veri tipi

- `window_stats(img, win)` → `(mean, std)` haritaları (her `(y, x)` sol üst köşeli pencere), iki integral görüntüyle; `std` için negatif varyansı 0'a kırp.
- `min_dtype(H, W, bits=8)` — $255\cdot H\cdot W$ (ve kare integral için $255^2HW$) toplamını taşıyabilen en küçük işaretsiz NumPy tamsayı tipi: `(dtype_ii, dtype_sq)` `np.uint16`, `np.uint32`, `np.uint64` arasından.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def window_stats(img, win):
    ...


def min_dtype(H, W, bits=8):
    ...


from skimage import data
m, s = window_stats(data.camera(), 24)
print(m.shape, s.max())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Öznitelik sayısını say: 162 336 nereden geliyor?

- `count_type(W, H, nx, ny)` — $nx\times ny$ temel blok yapısındaki (2-x için $nx=2, ny=1$, …) bir öznitelik türünün $W\times H$ penceredeki konum × boyut sayısı: genişlik $nx$'in, yükseklik $ny$'nin katı.
- `total_features(W, H)` — beş türün toplamı. 24 × 24 → 162 336; scikit-image'in `haar_like_feature_coord` sayılarıyla tür tür aynı olmalı.

In [ ]:
%%writefile extra_x2.py
from skimage.feature import haar_like_feature_coord


def count_type(W, H, nx, ny):
    ...


def total_features(W, H):
    ...


print(total_features(24, 24))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Kaskad aritmetiği ve ROC eğrisi

- `cascade_rates(stages)` — `[(d_i, f_i), …]` → `(D, F)`.
- `stages_needed(f_i, F_target)` — $f_i^n\le F$ için en küçük $n$.
- `roc(scores, y)` → `(fpr, tpr, auc)`: skor eşiklerini azalan sırada tara (eşit skorlar birlikte), yamuk kuralıyla AUC; `sklearn.metrics.roc_auc_score` ile aynı.

In [ ]:
%%writefile extra_x3.py
import math
import numpy as np


def cascade_rates(stages):
    ...


def stages_needed(f_i, F_target):
    ...


def roc(scores, y):
    ...


print(cascade_rates([(0.99, 0.3)] * 10))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Viola–Jones yüz dedektörünü baştan sona eğit

[Viola & Jones — Rapid Object Detection using a Boosted Cascade of Simple Features (CVPR 2001)](https://doi.org/10.1109/CVPR.2001.990517)

Bu modülün parçalarıyla gerçek bir dedektör: büyük bir yüz kümesi (ör. LFW, CelebA kırpılmış 24 × 24) ve yüz içermeyen görüntülerden binlerce negatif; makaledeki kaskad eğitim algoritması (her aşamada hedef $f_i$, $d_i$; negatifleri yalnız önceki aşamaları geçen yanlış pozitiflerden topla), çok ölçekli tarama (öznitelikleri 1.25 ile büyüterek) ve tespitleri birleştirme. FDDB ya da WIDER FACE'in kolay alt kümesinde değerlendir.


**Adımlar**

1. AdaBoost'u 162 336 öznitelikle hızlı tutmak için: öznitelikleri bir kez sırala ve sıralı indeksleri sakla (d aşamasındaki fikir), bellek için float16.
2. Aşama eşiği: doğrulama yüzlerinin %99.5'ini geçirecek kadar düşür.
3. Zor negatif madenciliği: önceki aşamaları geçen pencereler.
4. Tespit birleştirme: örtüşen kutuları ortalama (makale) ya da NMS (M4.9).

**Kabul ölçütleri**

- En az 10 aşamalı kaskad; aşama başına öznitelik sayısı, d_i ve f_i tablosu.
- 640 × 480 görüntüde CPU'da 10 fps üstü (vektörel tarama).
- ROC (tespit oranı – yanlış pozitif sayısı) ve OpenCV'nin hazır `haarcascade_frontalface_default.xml`'i ile karşılaştırma.

## Kaynaklar

- [Viola & Jones — Rapid Object Detection using a Boosted Cascade of Simple Features (CVPR 2001)](https://doi.org/10.1109/CVPR.2001.990517) — ©
- [Freund & Schapire — A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting (1997)](https://doi.org/10.1006/jcss.1997.1504) — ©
- [scikit-image — haar_like_feature, lfw_subset](https://scikit-image.org/docs/stable/auto_examples/applications/plot_haar_extraction_selection_classification.html) — BSD-3